# Tools + Skills

> Tools let a model **do** things. Skills teach it **how your team does them**.

**In this notebook**

1. What skills are, and how they differ from tools
2. A skill system built by hand: a folder of Markdown files and two small tools
3. Two examples, each run with the same tools, without and then with skills
4. Checks written in code, so the difference is measured against the policy
5. When a skill should *not* load

Continues from `llm_tools_01.ipynb`.

## What are skills?

A **skill** is a folder of instructions the model can read when a task calls for them. Its core is a `SKILL.md` file:

```text
skills/
└── expense-review/
    ├── SKILL.md            name, description and the procedure
    └── references/
        └── limits.md       details, read only if needed
```

A skill contains no new ability. It is text, and the model reads it using a tool. In this notebook, the skill's rules are enforced only by the model following them.

| | Tool | Skill |
|---|---|---|
| **What it is** | A function your code runs | Instructions the model reads |
| **What it gives** | An ability: look up, calculate, write | Know-how: when, in what order, by what rules |
| **Example** | `create_ticket(...)` | "Tickets for orders 7+ days late are high priority" |

## Why not put everything in the system prompt?

You could, for one procedure. With dozens, every request would pay for instructions it doesn't need. Skills load in stages, which is called **progressive disclosure**:

| Stage | What the model sees | When |
|---|---|---|
| 1. Catalog | Each skill's name and description | Always |
| 2. Procedure | The full `SKILL.md` | When the model decides the skill applies |
| 3. References | Extra files inside the skill | When the procedure calls for them |

```text
   question ─▶ model reads the catalog ─▶ load_skill("late-order")
                                                  │
                                                  ▼
   answer ◀─ business tools ◀─ follows the procedure, may read references
```

## Setup

Needs `OPENAI_API_KEY` in the repo-root `.env`. The optional `OPENAI_MODEL` picks the model.

In [1]:
import ast
import html
import json
import operator
import os
import re
import tempfile
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from IPython.display import HTML, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))

MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.4-mini")
BASE_INSTRUCTIONS = "You are a company support assistant. Answer concisely in plain text, without Markdown or LaTeX."

client = OpenAI()
print(f"Model: {MODEL}")

Model: gpt-5.4-mini


### The tool loop

The same loop as `llm_tools_01.ipynb`, with one change: the instructions are now a parameter, because that is where the skill catalog goes.

In [2]:
def run_agent(question, instructions, tools, registry, max_turns=10):
    """Returns (answer, trace); registry maps each tool name to its Python function."""
    conversation = [{"role": "user", "content": question}]
    trace = []

    for _ in range(max_turns):
        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=conversation,
            tools=tools,
        )
        conversation += response.output

        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text, trace

        for call in calls:
            arguments = json.loads(call.arguments)
            try:
                result = registry[call.name](**arguments)
            except Exception as error:
                # The model sees the failure and can retry or explain it.
                result = {"error": str(error)}
            trace.append({"tool": call.name, "arguments": arguments, "result": result})
            conversation.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

    raise RuntimeError(f"No final answer after {max_turns} turns")


def function_tool(name, description, properties):
    """Describes a function to the model; every property is a required argument."""
    return {
        "type": "function",
        "name": name,
        "description": description,
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": properties,
            "required": list(properties),
            "additionalProperties": False,
        },
    }

### Display helpers

In [3]:
CARD = (
    "font-family:system-ui,sans-serif;font-size:14px;line-height:1.5;color:#1f2933;"
    "background:{bg};border-left:4px solid {accent};border-radius:6px;"
    "padding:10px 14px;margin:8px 0;white-space:pre-wrap"
)
SKILL_TOOLS = {"load_skill", "read_skill_file"}


def card(title, body, bg, accent):
    style = CARD.format(bg=bg, accent=accent)
    display(HTML(f'<div style="{style}"><b style="color:{accent}">{html.escape(title)}</b>\n{html.escape(body)}</div>'))


def show_question(question):
    card("Question", question, "#eef2f7", "#52606d")


def show_answer(title, answer):
    card(title, answer, "#ecf8f1", "#15803d")


def show_trace(trace):
    if not trace:
        card("Tool calls", "None: the model answered directly.", "#eef2ff", "#4338ca")
    for step, entry in enumerate(trace, start=1):
        arguments = ", ".join(f"{name}={value!r}" for name, value in entry["arguments"].items())
        heading = f"Step {step}: {entry['tool']}({arguments})"
        if entry["tool"] in SKILL_TOOLS:
            # Skill text is printed once in the cell that writes it.
            card(heading, "returned the skill's text", "#f5f0ff", "#7c3aed")
        else:
            card(heading, f"returned {json.dumps(entry['result'], ensure_ascii=False)}", "#eef2ff", "#4338ca")


def show_checks(title, checks):
    """checks maps each policy rule to whether the run followed it."""
    lines = [f"{'PASS' if passed else 'FAIL'}  {rule}" for rule, passed in checks.items()]
    score = f"{sum(checks.values())} of {len(checks)} rules followed"
    all_passed = all(checks.values())
    card(f"{title}: {score}", "\n".join(lines), "#ecf8f1" if all_passed else "#fdf1ee", "#15803d" if all_passed else "#c2410c")

---
# The tools

These are the business tools. **Both runs of every example get exactly these**, so any difference comes from the skills.

The orders, tickets and vouchers are **fake data**, created for this notebook.

In [4]:
ORDERS = {
    "A1042": {"item": "Mechanical keyboard", "status": "in transit", "last_location": "Madrid sorting hub", "days_late": 7},
    "A1043": {"item": "USB-C cable", "status": "in transit", "last_location": "Porto depot", "days_late": 1},
}
TICKETS = []
VOUCHERS = []


def reset_store():
    TICKETS.clear()
    VOUCHERS.clear()


def get_order(order_id):
    return ORDERS.get(order_id, {"error": f"No order {order_id}"})


def create_ticket(order_id, priority, summary):
    ticket = {"ticket_id": f"T-{len(TICKETS) + 1:04d}", "order_id": order_id, "priority": priority, "summary": summary}
    TICKETS.append(ticket)
    return ticket


def issue_voucher(order_id, amount_usd):
    voucher = {"voucher_id": f"V-{len(VOUCHERS) + 1:04d}", "order_id": order_id, "amount_usd": amount_usd}
    VOUCHERS.append(voucher)
    return voucher


OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def evaluate(node):
    match node:
        case ast.Constant(value=int() | float() as value):
            return value
        case ast.BinOp(left=left, op=op, right=right) if type(op) in OPERATORS:
            return OPERATORS[type(op)](evaluate(left), evaluate(right))
    raise ValueError("Only numbers and + - * / are allowed")


def calculate(expression):
    return {"expression": expression, "value": evaluate(ast.parse(expression, mode="eval").body)}


business_tools = [
    function_tool(
        "get_order",
        "Look up an order's status, location and days late.",
        {"order_id": {"type": "string"}},
    ),
    function_tool(
        "create_ticket",
        "Open a support ticket about an order.",
        {
            "order_id": {"type": "string"},
            "priority": {"type": "string", "enum": ["normal", "high"]},
            "summary": {"type": "string"},
        },
    ),
    function_tool(
        "issue_voucher",
        "Give the customer a voucher for an order.",
        {"order_id": {"type": "string"}, "amount_usd": {"type": "number"}},
    ),
    function_tool(
        "calculate",
        "Evaluate an arithmetic expression exactly. Supports + - * / and parentheses.",
        {"expression": {"type": "string"}},
    ),
]
business_registry = {
    "get_order": get_order,
    "create_ticket": create_ticket,
    "issue_voucher": issue_voucher,
    "calculate": calculate,
}

---
# The skills

The skills are real files on disk, written to a temporary folder here so you can read them in the notebook.

In [5]:
SKILLS_DIR = Path(tempfile.mkdtemp()) / "skills"

SKILL_FILES = {
    "late-order/SKILL.md": """---
name: late-order
description: Handle a customer complaint about an order that has not arrived.
---
1. Look up the order with get_order.
2. Act on days_late:
   - 0 to 2: open no ticket and give no voucher. Tell the customer the order is on its way.
   - 3 to 6: open a ticket with priority normal.
   - 7 or more: open a ticket with priority high and issue a voucher for exactly 10 USD.
3. Every ticket summary must have exactly this format:
   LATE <order_id>: <days_late> days, last seen <last_location>
4. End your reply with the line: REF: <ticket_id>, or REF: none if no ticket was opened.
""",
    "expense-review/SKILL.md": """---
name: expense-review
description: Review travel expenses and work out how much is reimbursed.
---
1. Read references/limits.md with read_skill_file.
2. Apply the limits to each expense. Anything above a limit is not reimbursed.
3. Add up the reimbursed amount with the calculate tool, not in your head.
4. End your reply with the line: REIMBURSE: $<amount with two decimals>
""",
    "expense-review/references/limits.md": """Reimbursement limits:
- Hotel: 180 USD per night
- Meals: 60 USD per meal
- Taxi: no limit
""",
}

for relative_path, text in SKILL_FILES.items():
    path = SKILLS_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)

for relative_path, text in SKILL_FILES.items():
    card(f"skills/{relative_path}", text, "#f5f0ff", "#7c3aed")

## The skill system

The whole system takes three parts:

1. **`build_catalog`** reads each skill's name and description for the instructions (stage 1).
2. **`load_skill`** returns a full `SKILL.md` (stage 2).
3. **`read_skill_file`** returns a file inside a skill (stage 3).

In [6]:
def read_frontmatter(skill_file):
    header = skill_file.read_text().split("---")[1]
    return dict(line.split(": ", 1) for line in header.strip().splitlines())


def build_catalog():
    entries = [read_frontmatter(path) for path in sorted(SKILLS_DIR.glob("*/SKILL.md"))]
    return "\n".join(f"- {entry['name']}: {entry['description']}" for entry in entries)


def read_inside_skills(relative_path):
    # The path comes from the model, so it must not escape the skills folder.
    path = (SKILLS_DIR / relative_path).resolve()
    if not path.is_relative_to(SKILLS_DIR.resolve()) or not path.is_file():
        return {"error": f"No such skill file: {relative_path}"}
    return {"content": path.read_text()}


def load_skill(name):
    return read_inside_skills(f"{name}/SKILL.md")


def read_skill_file(skill, path):
    return read_inside_skills(f"{skill}/{path}")


skill_tools = [
    function_tool(
        "load_skill",
        "Load the full instructions of a skill from the catalog.",
        {"name": {"type": "string"}},
    ),
    function_tool(
        "read_skill_file",
        "Read a file inside a skill, when its instructions refer to one.",
        {
            "skill": {"type": "string"},
            "path": {"type": "string", "description": "Path inside the skill, e.g. 'references/limits.md'"},
        },
    ),
]
skill_registry = {"load_skill": load_skill, "read_skill_file": read_skill_file}

SKILLED_INSTRUCTIONS = f"""{BASE_INSTRUCTIONS}

Skills available:
{build_catalog()}

If a skill matches the task, call load_skill first and follow its instructions exactly."""

card("What the model always sees (stage 1)", SKILLED_INSTRUCTIONS, "#f5f0ff", "#7c3aed")

The two ways to run the agent differ only in the skill catalog and the two skill tools:

In [7]:
def without_skills(question):
    return run_agent(question, BASE_INSTRUCTIONS, business_tools, business_registry)


def with_skills(question):
    return run_agent(
        question,
        SKILLED_INSTRUCTIONS,
        business_tools + skill_tools,
        business_registry | skill_registry,
    )


def tools_called(trace):
    return [entry["tool"] for entry in trace]

---
# Example 1: A late order

**A procedure with company rules.** The tools can already do everything needed. What they can't supply is which ticket priority to use, whether a voucher is allowed, how much it is worth, or how the ticket must be written.

## Example

In [8]:
question = "My order A1042 still hasn't arrived. Can you sort this out?"
show_question(question)


def check_late_order():
    """Compares the store against the late-order skill for A1042, which is 7 days late."""
    ticket = TICKETS[0] if TICKETS else {}
    return {
        "Exactly one ticket opened": len(TICKETS) == 1,
        "Ticket priority is high": ticket.get("priority") == "high",
        "Ticket summary uses the required format": ticket.get("summary") == "LATE A1042: 7 days, last seen Madrid sorting hub",
        "Voucher of exactly 10 USD issued": [voucher["amount_usd"] for voucher in VOUCHERS] == [10],
    }

## Tools only, no skill

In [9]:
reset_store()
answer, trace = without_skills(question)

show_trace(trace)
show_answer("Tools only", answer)
show_checks("Tools only", check_late_order())

## Tools + skill

In [10]:
reset_store()
answer, trace = with_skills(question)
late_order_trace = trace

show_trace(trace)
show_answer("Tools + skill", answer)
show_checks("Tools + skill", check_late_order())
print("Tickets:", TICKETS)
print("Vouchers:", VOUCHERS)

Tickets: [{'ticket_id': 'T-0001', 'order_id': 'A1042', 'priority': 'high', 'summary': 'LATE A1042: 7 days, last seen Madrid sorting hub'}]
Vouchers: [{'voucher_id': 'V-0001', 'order_id': 'A1042', 'amount_usd': 10}]


### The same skill, a different branch

Order A1043 is only 1 day late. The skill says to open no ticket and give no voucher, so nothing should change.

In [11]:
reset_store()
question = "My order A1043 still hasn't arrived. Can you sort this out?"
show_question(question)

answer, trace = with_skills(question)

show_trace(trace)
show_answer("Tools + skill", answer)
show_checks("Tools + skill", {
    "No ticket opened": not TICKETS,
    "No voucher issued": not VOUCHERS,
    "Reply ends with REF: none": answer.strip().endswith("REF: none"),
})

## Explain

- **Tools only:** the model looked up the order, then stopped and *offered* to open a ticket and arrange a voucher. Nothing told it that it was allowed to act, so it scored 0 of 4. A different run might open a ticket, but the priority, format and voucher amount would still be guesses.
- **Tools + skill:** with the same tools, the model picked `late-order` from its description, loaded it, and then followed each step. The ticket summary matches the required format character for character.
- **The skill changed the behavior, not the abilities.** `create_ticket` and `issue_voucher` were available in both runs.
- **The A1043 run shows a rule, not a script.** The same skill led to a different result: 1 day late meant no ticket and no voucher. The model read the data and took the right branch.
- **The checks are ordinary Python** reading `TICKETS` and `VOUCHERS`, so they test what actually happened, not what the model says happened.

---
# Example 2: An expense review

**A rule the model can't guess.** This skill keeps its limits in a separate reference file, so the model has to take all three stages: catalog, procedure, reference. The company's limits exist nowhere else, so no model could know them.

## Example

In [12]:
question = (
    "Please review my Madrid trip expenses: hotel for 3 nights at $210 per night, "
    "one dinner at $95 and an airport taxi at $48. How much will I get reimbursed?"
)
show_question(question)

# Worked out from the limits file, with no model involved.
expected = 3 * min(210, 180) + min(95, 60) + 48


def check_expenses(answer):
    final_line = answer.strip().splitlines()[-1]
    return {
        f"Amount is ${expected:.2f}": f"{expected:.2f}" in answer or f"{expected:,}" in answer,
        "Reply ends with the REIMBURSE line": re.fullmatch(rf"REIMBURSE: \${expected}\.00", final_line.strip()) is not None,
    }


card("Expected from the policy", f"3 x $180 + $60 + $48 = ${expected:.2f}", "#eef2f7", "#52606d")

## Tools only, no skill

In [13]:
answer, trace = without_skills(question)

show_trace(trace)
show_answer("Tools only", answer)
show_checks("Tools only", check_expenses(answer))

## Tools + skill

In [14]:
answer, trace = with_skills(question)
expense_trace = trace

show_trace(trace)
show_answer("Tools + skill", answer)
show_checks("Tools + skill", check_expenses(answer))

## Explain

- **Tools only:** the model correctly added up everything and confidently said *"You'll be reimbursed $773."* The arithmetic is right and the answer is wrong, because it never knew about the limits, and it didn't say it was assuming there were none.
- **Tools + skill:** the trace shows all three stages. The model picked the skill from the catalog, read the procedure, read `references/limits.md`, and passed the capped amounts to `calculate`: **$648.00**, which matches the value computed from the policy.
- **Skills and tools each did their part.** The skill supplied the limits and the steps. The calculator supplied exact arithmetic, because the skill said to use it.
- **The right skill was chosen each time.** The model saw two skills in both examples and picked the matching one from its description.

---
# When a skill should not load

A skill costs tokens and time when it loads. Here the model has every skill available and a question none of them covers.

In [15]:
question = "What is the capital of France?"
show_question(question)

answer, trace = with_skills(question)

show_trace(trace)
show_answer("Tools + skill", answer)

The model loaded **no skill** and called no tool. Stage 1 cost only the few lines of the catalog, and the procedures and limits were never read.

---
# Summary

In [16]:
cell = "padding:6px 12px;border-bottom:1px solid #cbd2d9;text-align:left"
rows = [("Late order", late_order_trace), ("Expense review", expense_trace)]
header = "".join(f'<th style="{cell}">{title}</th>' for title in ("Example", "Tool calls with skills, in order"))
body = "".join(
    f'<tr><td style="{cell}">{name}</td><td style="{cell}"><code>{" → ".join(tools_called(trace))}</code></td></tr>'
    for name, trace in rows
)
display(HTML(f'<table style="border-collapse:collapse;font-family:system-ui,sans-serif"><tr>{header}</tr>{body}</table>'))

Example,"Tool calls with skills, in order"
Late order,load_skill → get_order → create_ticket → issue_voucher
Expense review,load_skill → read_skill_file → calculate


**What this does and doesn't show.** These outputs come from one saved run. They show that a model with tools and skills *can* follow a written procedure exactly. They don't measure how often it does, which would take many runs across many cases.

## Takeaways

1. **Tools give ability, skills give know-how.** The model needed both: the skill said what to do, the tools did it.
2. **A skill is just text plus a way to read it.** This notebook's whole system is a folder and two small functions.
3. **Progressive disclosure keeps context small.** Only names and descriptions are always loaded, so a large library stays cheap.
4. **The description decides.** The model picks a skill from its description alone, so write it for that choice.
5. **Skills guide, code enforces.** A skill is read and followed by the model, which can get it wrong. If a voucher must never exceed 10 USD, also check that inside `issue_voucher`.
6. **Skills are instructions, so trust matters.** A skill's text steers the model, so only load skills from sources you trust.